In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import einops
from fancy_einsum import einsum
import tqdm.auto as tqdm
import random
from pathlib import Path
import plotly.express as px
from torch.utils.data import DataLoader
import pickle

from jaxtyping import Float, Int
from typing import List, Union, Optional
from functools import partial
import copy

import itertools
from transformers import AutoModelForCausalLM, AutoConfig, AutoTokenizer
import dataclasses
import datasets
from IPython.display import HTML

# %%
import circuitsvis as cv

# %%
import transformer_lens
import transformer_lens.utils as utils
from transformer_lens.hook_points import (
    HookedRootModule,
    HookPoint,
)  # Hooking utilities
from transformer_lens import HookedTransformer, HookedTransformerConfig, FactoredMatrix, ActivationCache
from torch import Tensor
import io

# %%
torch.set_grad_enabled(False)

# %%
device = 'cuda:0'

In [2]:
def loadTransformerLensModel(modelPath):
    tokenizer = AutoTokenizer.from_pretrained(modelPath)
    hf_model = AutoModelForCausalLM.from_pretrained(modelPath, low_cpu_mem_usage=True)
    model = HookedTransformer.from_pretrained("meta-llama/Llama-2-7b-hf", hf_model=hf_model, device='cpu', fold_ln=False, center_writing_weights=False, center_unembed=False, tokenizer=tokenizer)

    return model, tokenizer

# %%
MODEL_PATH = "meta-llama/Llama-2-7b-hf"
model, tokenizer = loadTransformerLensModel(MODEL_PATH)
model = model.to(device)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded pretrained model meta-llama/Llama-2-7b-hf into HookedTransformer
Moving model to device:  cuda:0


In [3]:
with open('writing_heads_0.pkl', 'rb') as f:
    writing_heads = pickle.load(f)

In [4]:
writing_heads[0][:10]

[{'layer': 16,
  'head': 24,
  'input_id': tensor(29956, device='cuda:0'),
  'pos': -1,
  'prob': 0.5274432301521301},
 {'layer': 16,
  'head': 19,
  'input_id': tensor(29956, device='cuda:0'),
  'pos': -1,
  'prob': 0.16000570356845856},
 {'layer': 17,
  'head': 11,
  'input_id': tensor(29956, device='cuda:0'),
  'pos': -1,
  'prob': 0.10736586153507233},
 {'layer': 30,
  'head': 12,
  'input_id': tensor(29956, device='cuda:0'),
  'pos': -1,
  'prob': 0.1011650413274765},
 {'layer': 20,
  'head': 28,
  'input_id': tensor(29956, device='cuda:0'),
  'pos': -1,
  'prob': 0.08151783049106598},
 {'layer': 19,
  'head': 4,
  'input_id': tensor(29956, device='cuda:0'),
  'pos': -1,
  'prob': 0.07155190408229828},
 {'layer': 31,
  'head': 27,
  'input_id': tensor(29956, device='cuda:0'),
  'pos': -1,
  'prob': 0.050893671810626984},
 {'layer': 18,
  'head': 30,
  'input_id': tensor(29956, device='cuda:0'),
  'pos': -1,
  'prob': 0.03161605820059776},
 {'layer': 30,
  'head': 24,
  'input_id':

In [5]:
top_k_heads = []
bottom_k_heads = []
k = 10
for heads in writing_heads:
    for head in heads[:k]:
        top_k_heads.append((head['layer'], head['head']))
    for head in heads[-k:]:
        bottom_k_heads.append((head['layer'], head['head']))

In [6]:
top_k_heads = set(top_k_heads)
bottom_k_heads = set(bottom_k_heads)

In [7]:
len(top_k_heads), len(bottom_k_heads)

(46, 43)

In [8]:
import json
with open('/home/t-josingh/How-To-Think-Step-by-Step/data/activationPatching_llama2_clean.json', 'r') as f:
    COTData = json.load(f)

# noise_index = int(args.noiseIndex)
# number_of_examples = int(args.examples)
noise_index = 0
number_of_examples = 100


In [9]:
def getInputId(prompt):
    encoded_prompt =tokenizer.encode(prompt, add_special_tokens=False, return_tensors="pt")
    return encoded_prompt

In [10]:
def getCacheLogits(input_id):
    patched_cache = {}
    list_fwd_hooks = []
    def storeHookCache(value, hook):
        patched_cache[hook.name] = torch.from_numpy(value.detach().cpu().numpy())
    for layer in range(32):

        list_fwd_hooks.append((utils.get_act_name("z", layer, "attn"), storeHookCache))
            
    patched_logits = model.run_with_hooks(
            input_id, 
            fwd_hooks = list_fwd_hooks, 
            return_type="logits"
        )
    return patched_logits, patched_cache

In [11]:
import sys
sys.path.append('../')
from utilsFile.fewShotConstants import TemplateCOT_fictional, TemplateCOT_false

In [12]:
print(f"Noise index: {noise_index}")
print(f"Number of examples: {number_of_examples}")
prompts = [TemplateCOT_fictional.format(data['prompt']) + data[f'response_{noise_index}'] for data in COTData][:number_of_examples]


Noise index: 0
Number of examples: 100


In [13]:
input_id = getInputId(prompts[0])
logit, cache = getCacheLogits(input_id)

In [14]:
token_X_id = torch.argmax(logit[:, -1, :], dim=1)[0]
# initial_layer_head = stage_I(-1, token_X_id, 32)

In [15]:
probability = torch.nn.functional.softmax(logit[:, -1, :], dim=1)[0, token_X_id].item()

In [50]:
scaleFactor = 0

In [51]:
def scale_attention_heads(
        clean_head_vector: Float[torch.Tensor, "batch pos head_index d_head"],
        hook,
        head_index):
    clean_head_vector[:, :, head_index, :] = torch.mul(clean_head_vector[:, :, head_index, :], scaleFactor)
    return clean_head_vector

In [52]:
def scaleAttentionWeights(input_id, attentionHeadsList):
    list_fwd_hooks = []

    for layer in range(32):
        for head in range(32):
            if((layer, head) in attentionHeadsList):
                list_fwd_hooks.append((utils.get_act_name("z", layer, "attn"), partial(scale_attention_heads, head_index=head)))
    scaled_logits = model.run_with_hooks(
            input_id, 
            fwd_hooks = list_fwd_hooks, 
            return_type="logits"
        )
    return scaled_logits

In [53]:
scaled_logits = scaleAttentionWeights(input_id, bottom_k_heads)

In [54]:
scaled_probability = torch.nn.functional.softmax(scaled_logits[:, -1, :], dim=1)[0, token_X_id].item()

In [55]:
scaled_probability

0.9333898425102234

In [56]:
def getProbability(logits, token_X_id):
    return torch.nn.functional.softmax(logits[:, -1, :], dim=1)[0, token_X_id].item()

In [57]:
scaled_probability = getProbability(scaled_logits, token_X_id)
probability = getProbability(logit, token_X_id)

In [58]:
scaled_probability, probability

(0.9333898425102234, 0.9431114196777344)

In [59]:
from tqdm import tqdm
average_normalized_probability = []
progressBar = tqdm(prompts, desc=f"Noise_Index {noise_index}")
proabability_list = []
for prompt in progressBar:
    input_id = getInputId(prompt)
    logit, cache = getCacheLogits(input_id)
    token_X_id = torch.argmax(logit[:, -1, :], dim=1)[0]
    scaled_logits = scaleAttentionWeights(input_id, bottom_k_heads)
    probability = getProbability(logit, token_X_id)
    scaled_probability = getProbability(scaled_logits, token_X_id)
    proabability_list.append({'probability': probability, 'scaled_probability': scaled_probability})
    
    
    
    normalized_probability = (scaled_probability - probability) / probability
    average_normalized_probability.append(normalized_probability)
    
    progressBar.set_description(f"average_normalized_probability: {sum(average_normalized_probability) / len(average_normalized_probability)}")
    
    
print(f"Average Normalized Probability: {sum(average_normalized_probability) / len(average_normalized_probability)}")
with open(f'Noise_index_{noise_index}_scale_{scaleFactor}_bottom_k_heads_probability.json', 'w') as f:
    json.dump(proabability_list, f)
    
    

average_normalized_probability: 0.005970286116711532:  43%|████▎     | 43/100 [00:48<01:04,  1.13s/it]  


KeyboardInterrupt: 

In [60]:
scaleFactor

0